In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
from hydra import compose, initialize
from hydra.utils import instantiate
from omegaconf import open_dict
from sklearn.decomposition import PCA
from tqdm.auto import tqdm

from boa.model.module import ChgLightningModule

from mldft.ml.data.components.convert_transforms import ToTorch
from mldft.ml.data.components.of_data import Representation
from mldft.ml.models.components.loss_function import project_gradient
from mldft.utils.instantiators import instantiate_datamodule

In [2]:
device = "cuda"

with initialize(version_base=None, config_path="../configs"):
    config = compose(
        config_name="test.yaml",
        overrides=[
            "eval=qm9_pyscf_small",
            "data.datamodule.batch_size.val=1"
        ],
    )

basis_info = instantiate(config.data.basis_info)
datamodule_config = config.data.datamodule.copy()

datamodule = instantiate(datamodule_config, _recursive_=False)
datamodule.setup(stage="fit")

val_set = datamodule.val_dataset
train_set = datamodule.train_dataset

In [3]:
# Load model and hooks in same cell to not hook multiple times
checkpoint_path = config.ckpt_path[0]
best_model_path = Path(config.ckpt_path[0]) / "best_model_path.txt"
if best_model_path.exists():
    # check if the best_model_path is different from the ckpt_path
    with open(best_model_path, "r") as f:
        best_ckpt_path = f.read().strip()
else:
    raise FileNotFoundError("Best model path not found")
lightning_module = ChgLightningModule.load_from_checkpoint(
    best_ckpt_path, map_location=device
)
lightning_module.eval()

inputs = []
outputs = []
def hook_fn(module, input, output):
    inputs.append(input[0].detach().cpu())
    outputs.append(output.detach().cpu())
lightning_module.model.boa_stack.blocks[2].message_pass.register_forward_hook(hook_fn)

outputs_dict = {}
atom_keys = lightning_module.model.boa_stack.blocks[2].message_pass.inverse_mult_atoms.keys()
for key in atom_keys:
    atom_key = key
    print(atom_key)
    outputs_dict[atom_key] = ([], [])
    def hook_fn_a(module, input, output, key=atom_key):
        outputs_dict[key][0].append(input[0].detach().cpu())
        outputs_dict[key][1].append(output.detach().cpu())
    lightning_module.model.boa_stack.blocks[2].message_pass.inverse_mult_atoms[atom_key].register_forward_hook(hook_fn_a)

Unique atom types: [1 6 7 8 9]
Using radial correction for GTOs.
Using radial correction for GTOs.
Using radial correction for GTOs.
Using radial correction for GTOs.
Using radial correction for GTOs.
self.linear_basis: False
1
6
7
8
9


In [4]:
# Clear hook storage before running
inputs.clear()
outputs.clear()
for atom_key in atom_keys:
    outputs_dict[atom_key] = ([], [])

sample = next(iter(datamodule.val_dataloader())).to(device)
with torch.no_grad():
    coeffs, edge_index = lightning_module.model(sample)

In [5]:
print(len(inputs), len(outputs))
print(inputs[0].shape, outputs[0].shape)
print("natoms", sample.atomic_numbers)

1 1
torch.Size([972, 32]) torch.Size([972, 32])
natoms tensor([8, 6, 6, 6, 6, 8, 6, 6, 8, 1, 1, 1, 1, 1, 1], device='cuda:0',
       dtype=torch.int32)


### Visualizing Single Message Passing
The hooks you defined above capture the **aggregated** messages (summed over all neighbors) because `MessagePassAttention` aggregates the messages using `index_add` before returning, and `InverseMultAtom` is applied to these aggregated values.

To visualize the message transformation for a **single edge** (e.g., from atom $b$ to atom $a$), we need to manually compute the message $m_{ab}$ using the node features and the overlap matrices, as the individual edge messages are not exposed by the module's output.

The code below demonstrates how to:
1. Select a specific edge (from atom 1 to atom 0).
2. Extract the features of the sending node ($h_b$).
3. Compute the overlap $o_{ab} = W^{ab} h_b$.
4. Compute the message in the receiving basis $m_{ab} = (W^{aa})^{-1} o_{ab}$.
5. Evaluate and plot both $h_b$ and $m_{ab}$ on the grid to compare them.

In [13]:
# Select sending (b) and receiving (a) atoms
atom_a_idx = 0
atom_b_idx = 1

# Find the edge index for b -> a in message_edge_index
edge_indices = sample.message_edge_index.cpu()
edge_idx = -1
for i in range(edge_indices.shape[1]):
    if edge_indices[0, i] == atom_a_idx and edge_indices[1, i] == atom_b_idx:
        edge_idx = i
        break

if edge_idx == -1:
    print(f"No edge found from atom {atom_b_idx} to atom {atom_a_idx}")
else:
    print(f"Analyzing edge {edge_idx}: Atom {atom_b_idx} -> Atom {atom_a_idx}")

    # 1. Get features of sending node b (h_b)
    # inputs[0] contains all node features
    node_features = inputs[0] # Shape: (total_basis_funcs, channels)
    mask_b = (coeff_ind_to_node_ind == atom_b_idx)
    h_b = node_features[mask_b] # Shape: (basis_dim_b, channels)
    
    # 2. Get overlap matrix W_ab
    # sample.message_edge_matrices is (num_edges, max_dim, max_dim)
    W_ab = sample.message_edge_matrices[edge_idx].cpu()
    
    # Determine actual basis dimensions
    mask_a = (coeff_ind_to_node_ind == atom_a_idx)
    h_a = node_features[mask_a]
    dim_a = h_a.shape[0]
    dim_b = h_b.shape[0]
    
    # Slice W_ab to actual dimensions
    W_ab = W_ab[:dim_a, :dim_b]
    
    # 3. Compute overlap o_ab = W_ab @ h_b
    o_ab = torch.matmul(W_ab, h_b) # Shape: (dim_a, channels)
    
    # 4. Get inverse overlap matrix of receiving node a: (W_aa)^-1
    atom_type_a = sample.atomic_numbers[atom_a_idx].item()
    inv_W_aa = lightning_module.model.boa_stack.inv_overlap_matrices[atom_type_a].cpu()
    
    # 5. Compute message m_ab = (W_aa)^-1 @ o_ab
    m_ab = torch.matmul(inv_W_aa, o_ab) # Shape: (dim_a, channels)
    
    # --- Visualization ---
    # Evaluate h_b (sending features) on the grid
    # ao[0] has shape (num_grid, total_num_basis)
    ao_b = ao[0, :, mask_b] # Shape: (num_grid, dim_b)
    val_h_b = ao_b @ h_b.numpy() # Shape: (num_grid, channels)
    
    # Evaluate m_ab (transformed message) on the grid
    # m_ab is in the basis of atom a
    ao_a = ao[0, :, mask_a] # Shape: (num_grid, dim_a)
    val_m_ab = ao_a @ m_ab.numpy() # Shape: (num_grid, channels)
    
    # Plotting
    channel_idx = 0 # Visualize first channel
    
    # Use the 2D grid coordinates for plotting
    # x and y are from the meshgrid defined earlier
    grid_x = x.flatten()
    grid_y = y.flatten()
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Plot h_b
    c1 = axes[0].tricontourf(grid_x, grid_y, val_h_b[:, channel_idx], levels=20)
    axes[0].set_title(f"Sending Feature $h_b$ (Atom {atom_b_idx})")
    plt.colorbar(c1, ax=axes[0])
    
    # Plot m_ab
    c2 = axes[1].tricontourf(grid_x, grid_y, val_m_ab[:, channel_idx], levels=20)
    axes[1].set_title(f"Transformed Message $m_{{ab}}$ (in Atom {atom_a_idx} Basis)")
    plt.colorbar(c2, ax=axes[1])
    
    # Plot Difference
    diff = val_h_b[:, channel_idx] - val_m_ab[:, channel_idx]
    c3 = axes[2].tricontourf(grid_x, grid_y, diff, levels=20)
    axes[2].set_title(f"Difference ($h_b - m_{{ab}}$)")
    plt.colorbar(c3, ax=axes[2])
    
    # Plot atom positions
    for ax in axes:
        ax.scatter(rot_pos_xy[:, 0], rot_pos_xy[:, 1], c='red', marker='o')
        ax.text(rot_pos_xy[atom_a_idx, 0], rot_pos_xy[atom_a_idx, 1], f"A{atom_a_idx}", color='white')
        ax.text(rot_pos_xy[atom_b_idx, 0], rot_pos_xy[atom_b_idx, 1], f"B{atom_b_idx}", color='white')

    plt.show()

Analyzing edge 21: Atom 1 -> Atom 0


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 83 is different from 10000)